## Post-Generation Validation

> 📖 Read the full article: [Structured Output Tools for LLMs: Instructor, PydanticAI, LangChain, Outlines,](https://codecut.ai/structured-llm-outputs-tools-comparison/)


These tools validate output after generation. Instructor retries on failure. PydanticAI and LangChain use API-level enforcement with Pydantic validation.

### Instructor: Simplest Integration

[Instructor](https://github.com/instructor-ai/instructor) (12.3k stars) wraps any LLM client with Pydantic validation and automatic retry.

Unlike PydanticAI's dependency injection or LangChain's ecosystem complexity, Instructor stays focused on one thing: structured outputs with minimal code.

To install Instructor, run:

```bash
pip install instructor
```

*This article uses instructor v1.14.4.*

To use Instructor:

- Define a Pydantic model with your desired fields
- Wrap your LLM client (OpenAI, Anthropic, Ollama, etc.) with Instructor
- Pass the model as `response_model` in your API call

The code below extracts sales lead information from an email:

In [ ]:
import instructor
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

class SalesLead(BaseModel):
    company_size: Literal["startup", "smb", "enterprise"]
    priority: Literal["low", "medium", "high"]

client = instructor.from_openai(OpenAI())

email = "Hi, I'm the CTO of a 500-person company. We're interested in your enterprise plan. Can we schedule a demo?"

lead = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": f"Extract sales lead info from this email: {email}"}],
    response_model=SalesLead,
    max_retries=3
)
print(lead)

Output:

```text
company_size='enterprise' priority='high'
```

Each field matches the schema: `company_size` and `priority` are constrained to the allowed `Literal` values.

The first LLM response may return an invalid value like "large" instead of "enterprise". When this happens, Instructor sends the validation error back for self-correction.

### PydanticAI: Type-Safe Agents

[PydanticAI](https://github.com/pydantic/pydantic-ai) (14.5k stars) brings FastAPI's developer experience to AI agents.

While Instructor focuses on extraction, PydanticAI supports tools and dependency injection. Tools are functions the agent can call to fetch external dat.


To install PydanticAI, run:

```bash
pip install pydantic-ai
```

*This article uses pydantic-ai v1.48.0.*

PydanticAI uses async internally. If running in a Jupyter notebook, apply `nest_asyncio` to avoid event loop conflicts:

In [ ]:
import nest_asyncio

nest_asyncio.apply()

For basic extraction, PydanticAI takes a different approach with an Agent abstraction, but the output resembles Instructor.

In [ ]:
from pydantic_ai import Agent
from pydantic import BaseModel
from typing import Literal

class SalesLead(BaseModel):
    company_size: Literal["startup", "smb", "enterprise"]
    priority: Literal["low", "medium", "high"]

agent = Agent("openai:gpt-4o", output_type=SalesLead)

email = "Hi, I'm the CTO of a 500-person company. We're interested in your enterprise plan. Can we schedule a demo?"

result = agent.run_sync(f"Extract sales lead info from this email: {email}")
print(result.output)

```text
company_size='enterprise' priority='high'
```

Where PydanticAI stands out is tools and dependency injection. Tools are functions the agent can call during generation to fetch external data. Dependency injection passes data into those tools without hardcoding values.

To use PydanticAI with tools and dependency injection:

- Create a dataclass for external data (e.g., pricing table)
- Add `deps_type` to the agent to specify the dependency class
- Decorate functions with `@agent.tool` to make them callable
- Provide dependencies when calling `run_sync()`

In [ ]:
from pydantic_ai import Agent, RunContext
from pydantic import BaseModel
from dataclasses import dataclass
from typing import Literal

class SalesLead(BaseModel):
    company_size: Literal["startup", "smb", "enterprise"]
    priority: Literal["low", "medium", "high"]
    monthly_price: int

@dataclass
class PricingTable:
    prices: dict[str, int]

agent = Agent(
    "openai:gpt-4o",
    deps_type=PricingTable,
    output_type=SalesLead
)

@agent.tool
def get_price(ctx: RunContext[PricingTable], company_size: str) -> str:
    """Get monthly price for a company size tier."""
    price = ctx.deps.prices.get(company_size.lower(), 0)
    return f"Monthly price for {company_size}: ${price}"

email = "Hi, I'm the CTO of a 500-person company. We're interested in your enterprise plan."

result = agent.run_sync(
    f"Extract sales lead info from this email: {email}",
    deps=PricingTable(prices={"startup": 99, "smb": 499, "enterprise": 1999})
)
print(result.output)

```text
company_size='enterprise' priority='high' monthly_price=1999
```

The output shows `monthly_price=1999`, which matches the enterprise tier in the `PricingTable`. The LLM called `get_price("enterprise")` to retrieve this value.

For a deeper dive into PydanticAI's capabilities, see [Enforce Structured Outputs from LLMs with PydanticAI](https://codecut.ai/enforce-structured-outputs-from-llms-with-pydanticai/).

### LangChain: Ecosystem Integration

[LangChain](https://github.com/langchain-ai/langchain) (125k stars) offers structured outputs as part of a comprehensive framework.

While Instructor and PydanticAI focus on extraction, LangChain provides structured outputs as part of a larger ecosystem. This includes integrations with vector stores, tools, and monitoring.

To install LangChain, run:

```bash
pip install langchain langchain-openai
```

*This article uses langchain v1.2.7 and langchain-openai v1.1.7.*

To use LangChain for structured outputs:

- Create a chat model (OpenAI, Anthropic, Google, etc.)
- Call `.with_structured_output(YourModel)` to add schema enforcement
- Use `.invoke()` with your prompt

The code below extracts sales lead information from an email:

In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal

class SalesLead(BaseModel):
    company_size: Literal["startup", "smb", "enterprise"]
    priority: Literal["low", "medium", "high"]

model = ChatOpenAI(model="gpt-4o")
structured = model.with_structured_output(SalesLead)

email = "Hi, I'm the CTO of a 500-person company. We're interested in your enterprise plan. Can we schedule a demo?"

lead = structured.invoke(f"Extract sales lead info from this email: {email}")
print(lead)

Output:

```text
company_size='enterprise' priority='high'
```

The output resembles Instructor and PydanticAI since all three use Pydantic models for schema enforcement.

LangChain's value is ecosystem integration. You can combine structured outputs with:

- Vector stores for RAG pipelines
- Document loaders for PDFs, web pages, and databases
- Memory for conversation history
- LangSmith for monitoring and tracing
- And many more integrations

### When to Use Each Tool

LangChain covers the most features, but I find the simpler tools easier to maintain when you don't need the full ecosystem.

- **Instructor**: One pip install, zero framework concepts. Choose when extraction is your only need.
- **PydanticAI**: Adds tools without the full LangChain ecosystem. Choose when you need external data but not RAG or memory.
- **LangChain**: Full ecosystem with learning curve. Choose when you're already using LangChain or need its integrations.

For production patterns like PII filtering and human approval workflows, see [Build Production-Ready LLM Agents with LangChain 1.0 Middleware](https://codecut.ai/langchain-1-0-middleware-production-agents/).

## Pre-Generation Constraints

Unlike post-generation validation tools that check output after generation, these tools guide the LLM character-by-character. Invalid characters are blocked before they're generated. This guarantees 100% schema compliance. No wasted API calls on invalid outputs.

### Outlines: Guaranteed Valid JSON

[Outlines](https://github.com/dottxt-ai/outlines) (13.3k stars) guarantees valid output by constraining token sampling during generation.

Among pre-generation constraint tools, Outlines is the simplest. 

To install Outlines, run:

```bash
pip install outlines
```

*This article uses outlines v1.2.9.*

The code resembles Instructor, but works differently. At each generation step, Outlines checks which tokens would keep the output valid and blocks all others. The model can only choose from schema-compliant tokens:

In [ ]:
import outlines
from transformers import AutoModelForCausalLM, AutoTokenizer
from pydantic import BaseModel
from typing import Literal

class SalesLead(BaseModel):
    company_size: Literal["startup", "smb", "enterprise"]
    priority: Literal["low", "medium", "high"]

# Load local model for direct token control
model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B"),
    AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")
)

email = "Hi, I'm the CTO of a 500-person company. We're interested in your enterprise plan."

result = model(
    f"Extract sales lead info from this email: {email}",
    SalesLead,
    max_new_tokens=100
)
print(result)

Output:

```text
company_size='enterprise' priority='high'
```

The `company_size` and `priority` fields contain valid `Literal` values. Invalid values are impossible because Outlines blocks those tokens during generation.

Beyond schema validation, Outlines supports regex and choice constraints that block invalid tokens during generation.

For example, this regex enforces a phone number format:

In [ ]:
result = model("New York office phone number:", output_type=Regex(r"\(\d{3}\) \d{3}-\d{4}"))
print(result)

Output:

```text
(212) 555-0147
```

Similarly, a Literal type restricts output to predefined values:

In [ ]:
Sentiment = Literal["positive", "negative", "neutral"]
result = model("The product exceeded expectations! Sentiment:", output_type=Sentiment)
print(result)

Output:

```text
positive
```

These constraints work at the token level: the model cannot generate invalid characters because they are blocked before generation.

### Guidance: Control Flow Within Generation

[Guidance](https://github.com/guidance-ai/guidance) (19k stars) lets you mix Python if/else with constrained generation.

Outlines constrains output format. Guidance goes further by letting Python if/else statements run during generation. The model's output becomes a variable you can check, then generation continues down the chosen branch.

To install Guidance, run:

```bash
pip install guidance
```

*This article uses guidance v0.3.0.*

With `@guidance`, you define functions where generation pauses for Python control flow:

- The function classifies an email as sales, support, or spam
- `lm["category"]` stores the model's constrained choice
- If "sales", generate a lead score and priority level
- If "support", generate a ticket with category and urgency
- If "spam", generation stops with no additional output

In [ ]:
from guidance import models, select, gen, guidance
from guidance import json as gen_json
from pydantic import BaseModel
from typing import Literal

class SalesLead(BaseModel):
    model_config = dict(extra="forbid")
    company_size: Literal["startup", "smb", "enterprise"]
    priority: Literal["low", "medium", "high"]

class SupportTicket(BaseModel):
    model_config = dict(extra="forbid")
    issue_type: Literal["billing", "technical", "account"]
    urgency: Literal["low", "medium", "high"]

lm = models.Transformers("Qwen/Qwen2.5-1.5B")

@guidance
def route_email(lm, email):
    lm += f"Email: {email}\n"
    lm += f"Category: {select(['sales', 'support', 'spam'], name='category')}\n"

    if lm["category"] == "sales":
        lm += gen_json(name="lead", schema=SalesLead)
    elif lm["category"] == "support":
        lm += gen_json(name="ticket", schema=SupportTicket)

    return lm

email = "Hi, I'm the CTO of a 500-person company. We're interested in your enterprise plan. Can we schedule a demo?"
result = lm + route_email(email)

if "lead" in result:
    print(f"Lead: {result['lead']}")
elif "ticket" in result:
    print(f"Ticket: {result['ticket']}")

While the model generates, Python control flow runs alongside it:

```text
Email: Hi, I'm the CTO of a 500-person company. We're interested in your enterprise plan. Can we schedule a demo?
Category: sales
{"company_size": "enterprise", "priority": "high"}
```
The output shows how branching works in real-time: once the model chose "sales", Guidance executed the `if` branch and generated the `SalesLead` JSON.

Output from the print statement:

```text
Lead: {"company_size": "enterprise", "priority": "high"}
```

### When to Use Each Tool

- **Outlines**: Choose when you need guaranteed schema compliance with straightforward extraction. Simpler API, easier to get started.
- **Guidance**: Choose when you need conditional logic during generation. The `@guidance` decorator lets Python control flow run alongside token generation.